In [1]:
import time
import pandas as pd
import winsound # Thư viện tạo tiếng Bíp báo hiệu trên Windows
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options

In [2]:
def crawl_1200_reviews(driver, url, address_name):
    print(f"\n[{address_name}] >>> Đang mở link...")
    try:
        driver.get(url)
    except Exception as e:
        print(f"Lỗi truy cập: {e}")
        return []

    time.sleep(6) 
    
    try:
        tab = driver.find_element(By.XPATH, '//button[.//div[contains(text(), "Bài đánh giá") or contains(text(), "Reviews")]]')
        tab.click()
        time.sleep(3)
    except:
        pass

    try: winsound.Beep(2000, 1200)
    except: pass
        
    print("\n" + "="*60)
    print(f"🚨 ĐÃ MỞ XONG LINK: {address_name}")
    print("👉 BƯỚC 1: Hãy qua cửa sổ Chrome, tự chọn bộ lọc (Mới nhất/Cao nhất/Thấp nhất).")
    print("👉 BƯỚC 2: Quay lại đây BẤM PHÍM ENTER để code quét.")
    print("="*60)
    
    input("Bấm ENTER tại đây khi bạn đã sẵn sàng...") 

    collected_data = []
    seen_texts = set()
    stuck_counter = 0
    last_total = 0

    print(f"\n🚀 Bắt đầu càn quét mục tiêu 1200 đánh giá (Chống lỗi bật popup, Đợi lâu hơn)...")

    while len(collected_data) < 1200: 
        # --- SỬA LỖI 1: TỰ ĐỘNG ĐÓNG POPUP ---
        # Gửi phím ESCAPE để thoát khỏi bất kỳ hình ảnh/hồ sơ nào lỡ bị bật lên che màn hình
        try:
            webdriver.ActionChains(driver).send_keys(Keys.ESCAPE).perform()
            time.sleep(0.5)
        except:
            pass
            
        items = driver.find_elements(By.CSS_SELECTOR, 'div[data-review-id]')
        if items:
            try:
                last_item = items[-1]
                driver.execute_script("arguments[0].scrollIntoView(true);", last_item)
                time.sleep(1)
                
                # BỎ LỆNH .click() để không ấn nhầm vào ảnh/avatar
                action = webdriver.ActionChains(driver)
                action.move_to_element(last_item).perform() 
                
                # Tăng số lần ấn Page Down để kéo sâu hơn
                for _ in range(4):
                    action.send_keys(Keys.PAGE_DOWN).perform()
                    time.sleep(0.5)
            except:
                driver.execute_script("""
                    let mainDivs = document.querySelectorAll('div[role="main"]');
                    if(mainDivs.length > 1) { mainDivs[1].scrollTop = mainDivs[1].scrollHeight; }
                """)
        
        # --- SỬA LỖI 2: TĂNG THỜI GIAN ĐỢI ---
        time.sleep(5) # Trễ 5 giây để Google Maps kịp kéo data mới về

        # --- ÉP BUNG FULL TEXT VÀ BẢN GỐC ---
        driver.execute_script("""
            let btns = document.querySelectorAll('button');
            for(let btn of btns){
                let txt = (btn.innerText || "").toLowerCase();
                if(txt.includes('xem thêm') || txt.includes('more') || 
                   txt.includes('bản gốc') || txt.includes('original') || 
                   btn.classList.contains('w8Bnuf')) {
                    try { btn.click(); } catch(e) {}
                }
            }
        """)
        
        # BẮT BUỘC CHỜ 3 GIÂY để DOM load đủ chữ
        time.sleep(3) 

        items = driver.find_elements(By.CSS_SELECTOR, 'div[data-review-id]')
        
        # Bắt đầu đọc dữ liệu
        for item in items:
            if len(collected_data) >= 1200:
                break
            try:
                detail_full = item.find_element(By.CLASS_NAME, "wiI7pd").text
                
                # Bỏ qua nếu text rỗng hoặc VẪN CÒN chứa "..."
                if not detail_full or detail_full.endswith("..."):
                    continue
                    
                if detail_full in seen_texts:
                    continue
                    
                rating_raw = item.find_element(By.XPATH, './/span[contains(@aria-label, "sao") or contains(@aria-label, "star")]').get_attribute("aria-label")
                rating = int(rating_raw.split()[0])
                user = item.find_element(By.CLASS_NAME, "d4r55").text
                
                seen_texts.add(detail_full)
                collected_data.append({
                    "Username": user,
                    "Address": address_name,
                    "Rating": rating,
                    "Detail": detail_full
                })
            except:
                continue

        current_total = len(collected_data)
        print(f"  -> Đã gom được {current_total}/1200 đánh giá (Bản gốc, Full chữ)...")

        if current_total >= 1200:
            print(f"🎉 TUYỆT VỜI! Đã cào đủ 1200 đánh giá nguyên bản!")
            break

        # --- SỬA LỖI 3: KIÊN NHẪN HƠN KHI BỊ KẸT ---
        if current_total == last_total:
            stuck_counter += 1
            if stuck_counter >= 15: # Cho phép kẹt tới 15 vòng (rất lâu) mới chốt sổ
                print("⚠️ Google Maps đã ngưng nhả dữ liệu. Sẽ chốt sổ với số lượng hiện tại.")
                break
        else:
            stuck_counter = 0

        last_total = current_total

    return collected_data

In [5]:
# ==========================================
# KHU VỰC CHẠY CHÍNH 
# ==========================================

# 🛑 ĐIỀN THÔNG TIN 1 ĐỊA ĐIỂM CỦA BẠN VÀO ĐÂY:
ten_dia_diem = "Po Nagar Cham Towers"
# NHỚ THAY ĐƯỜNG LINK THẬT DƯỚI ĐÂY NHÉ:
link_google_maps = "https://www.google.com/maps/place/Po+Nagar+Temple/@12.2653665,109.1927929,17z/data=!4m8!3m7!1s0x3170678c61b8f251:0x115f6f97f1af1d7c!8m2!3d12.2653665!4d109.1953678!9m1!1b1!16s%2Fm%2F02x5pgr?entry=ttu&g_ep=EgoyMDI2MDMxOC4xIKXMDSoASAFQAw%3D%3D"

options = Options()
options.add_argument("--disable-notifications") # Chỉ giữ lại lệnh tắt thông báo

driver = webdriver.Chrome(options=options)
driver.maximize_window() 

# Khởi chạy 1 lần duy nhất
all_raw_data = crawl_1200_reviews(driver, link_google_maps, ten_dia_diem)

driver.quit()

# ==========================================
# XUẤT FILE KHÔNG GIỚI HẠN
# ==========================================
print("\n📊 Đang tiến hành xóa trùng lặp và xuất file...")

if all_raw_data:
    df_all = pd.DataFrame(all_raw_data)
    # Giữ lại toàn bộ dữ liệu, chỉ xóa những dòng trùng nhau y hệt
    df_final = df_all.drop_duplicates(subset=['Address', 'Username', 'Detail'])

    # Đặt tên file xuất ra theo tên địa điểm cho dễ quản lý
    ten_file = f"Data_{ten_dia_diem.replace(' ', '_')}.csv"
    df_final.to_csv(ten_file, index=False, encoding='utf-8-sig')
    
    print(f"🎉 THÀNH CÔNG! Đã giữ lại trọn vẹn {len(df_final)} dòng ra file {ten_file}")
else:
    print("\n😢 Không lấy được dữ liệu nào.")


[Po Nagar Cham Towers] >>> Đang mở link...

🚨 ĐÃ MỞ XONG LINK: Po Nagar Cham Towers
👉 BƯỚC 1: Hãy qua cửa sổ Chrome, tự chọn bộ lọc (Mới nhất/Cao nhất/Thấp nhất).
👉 BƯỚC 2: Quay lại đây BẤM PHÍM ENTER để code quét.

🚀 Bắt đầu càn quét mục tiêu 1200 đánh giá (Chống lỗi bật popup, Đợi lâu hơn)...
  -> Đã gom được 20/1200 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 30/1200 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 40/1200 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 50/1200 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 60/1200 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 70/1200 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 80/1200 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 88/1200 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 97/1200 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 107/1200 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 117/1200 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 127/1200 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được